Dado que el entrenamiento de redes neuronales es una tarea  muy costosa, **se recomienda ejecutar el notebooks en [Google Colab](https://colab.research.google.com)**, por supuesto también se puede ejecutar en local.

Al entrar en [Google Colab](https://colab.research.google.com) bastará con hacer click en `upload` y subir este notebook. No olvide luego descargarlo en `File->Download .ipynb`

**El examen deberá ser entregado con las celdas ejecutadas, si alguna celda no está ejecutadas no se contará.**

El examen se divide en dos partes, con la puntuación que se indica a continuación. La puntuación máxima será 10.

- [Actividad 1: Redes Densas](#actividad_1): 5 pts
    - Correcta normalización: máximo de 0.25 pts
    - [Cuestión 1](#1.1): 1.5 pt
    - [Cuestión 2](#1.2): 1.5 pt
    - [Cuestión 3](#1.3): 0.5 pts
    - [Cuestión 4](#1.4): 0.25 pts
    - [Cuestión 5](#1.5): 0.25 pts
    - [Cuestión 6](#1.6): 0.25 pts
    - [Cuestión 7](#1.7): 0.25 pts
    - [Cuestión 8](#1.8): 0.25 pts


- [Actividad 2: Redes Convolucionales](#actividad_2): 5 pts
    - [Cuestión 1](#2.1): 2.5 pt
    - [Cuestión 2](#2.2): 1 pt
    - [Cuestión 3](#2.3): 0.5 pts
    - [Cuestión 4](#2.4): 0.5 pts
    - [Cuestión 5](#2.5): 0.5 pts
    

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, models
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras import Input
from tensorflow.keras import layers
tf.random.set_seed(0)

<a name='actividad_1'></a>
# Actividad 1: Redes Densas

Para esta primera actividad vamos a utilizar el [wine quality dataset](https://archive.ics.uci.edu/ml/datasets/wine+quality). Con el que trataremos de predecir la calidad del vino.

La calidad del vino puede tomar valores decimales (por ejemplo 7.25), independientemente de que en el dataset de entrenamiento sean números enteros. Por lo tanto, el problema es una `regresión`.

**Puntuación**: 

Normalizar las features correctamente (x_train, x_test): 0.25 pts , se pueden normalizar con el [Normalization layer](https://www.tensorflow.org/api_docs/python/tf/keras/layers/experimental/preprocessing/Normalization) de Keras. 


- Correcta normalización: máximo de 0.25 pts
- [Cuestión 1](#1.1): 1 pt
- [Cuestión 2](#1.2): 1 pt
- [Cuestión 3](#1.3): 0.5 pts
- [Cuestión 4](#1.4): 0.25 pts
- [Cuestión 5](#1.5): 0.25 pts
- [Cuestión 6](#1.6): 0.25 pts
- [Cuestión 7](#1.7): 0.25 pts
- [Cuestión 8](#1.8): 0.25 pts



In [ ]:
# Descargar los datos con pandas
df_red = pd.read_csv(
    'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv',
    sep=';'
)
df_white = pd.read_csv(
    'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv',
    sep=';'
)
df = pd.concat([df_red, df_white])

df.head()

In [ ]:
feature_names = [
    'fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 
    'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol'
]


# separar features y target
y = df.pop('quality').values
X = df.copy().values

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

print('x_train, y_train shapes:', x_train.shape, y_train.shape)
print('x_test, y_test shapes:', x_test.shape, y_test.shape)
print('Some qualities: ', y_train[:5])

In [ ]:
## Si quiere, puede normalizar las features
normalizer = layers.Normalization()
normalizer.adapt(x_train)  # Ajusta la normalización con los datos de entrenamiento

# Mostrar algunos datos normalizados para verificar
print("Datos de entrada normalizados:", normalizer(x_train[:5]))


<a name='1.1'></a>
## Cuestión 1: Cree un modelo secuencial que contenga 4 capas ocultas(hidden layers), con más de 60 neuronas  por capa, sin regularización y obtenga los resultados.

Puntuación: 
- Obtener el modelo correcto: 0.8 pts
- Compilar el modelo: 0.1pts
- Acertar con la función de pérdida: 0.1 pts

In [ ]:
# Crear el modelo secuencial con 4 capas ocultas
model = tf.keras.models.Sequential([
    normalizer, 
    layers.Dense(64, activation='relu', input_shape=(x_train.shape[1],)), 
    layers.Dense(64, activation='relu'),  
    layers.Dense(64, activation='relu'), 
    layers.Dense(64, activation='relu'), 
    layers.Dense(1) 
])


In [ ]:
# Compilación del modelo
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005), loss='mean_squared_error')

In [ ]:
# No modifique el código
model.fit(x_train,
          y_train,
          epochs=200,
          batch_size=32,
          validation_split=0.2,
          verbose=1)

In [ ]:
# No modifique el código
results = model.evaluate(x_test, y_test, verbose=1)
print('Test Loss: {}'.format(results))

<a name='1.2'></a>
## Cuestión 2: Utilice el mismo modelo de la cuestión anterior pero añadiendo al menos dos técnicas distinas de regularización.

Ejemplos de regularización: [Prevent_Overfitting.ipynb](https://github.com/ezponda/intro_deep_learning/blob/main/class/Fundamentals/Prevent_Overfitting.ipynb)

Puntuación:

- Obtener el modelo con la regularización: 0.8 pts
- Obtener un `test loss` inferior al anterior: 0.2 pts


In [ ]:
model = tf.keras.models.Sequential([
    normalizer,  # Normalizador de los datos
    layers.Dense(64, activation='relu', input_shape=(x_train.shape[1],), 
                  kernel_regularizer=regularizers.l1_l2(l1=0.005, l2=0.005)),
    layers.Dropout(0.1),  # Dropout con una tasa del 10% (más pequeño)
    layers.Dense(64, activation='relu', 
                 kernel_regularizer=regularizers.l1_l2(l1=0.005, l2=0.005)),
    layers.Dropout(0.1),  # Dropout con una tasa del 10%
    layers.Dense(64, activation='relu', 
                 kernel_regularizer=regularizers.l1_l2(l1=0.005, l2=0.005)), 
    layers.Dropout(0.1),
    layers.Dense(64, activation='relu', 
                 kernel_regularizer=regularizers.l1_l2(l1=0.005, l2=0.005)), 
    layers.Dense(1)
])


In [ ]:
# Compilación del modelo
# Código aquí
model.compile(optimizer='adam', loss='mean_squared_error')


In [ ]:

batch_size=32

In [ ]:
# No modifique el código
model.fit(x_train,
          y_train,
          epochs=200,
          batch_size=batch_size,
          validation_split=0.2,
          verbose=1)

In [ ]:
# No modifique el código
results = model.evaluate(x_test, y_test, verbose=1)
print('Test Loss: {}'.format(results))

<a name='1.3'></a>
## Cuestión 3: Utilice el mismo modelo de la cuestión anterior pero añadiendo un callback de early stopping.

In [ ]:
model = tf.keras.models.Sequential([
    normalizer, 
    layers.Dense(64, activation='relu', input_shape=(x_train.shape[1],), 
                 kernel_regularizer=regularizers.l1_l2(l1=0.005, l2=0.005)),
    layers.Dropout(0.1),  
    layers.Dense(64, activation='relu', 
                 kernel_regularizer=regularizers.l1_l2(l1=0.005, l2=0.005)),
    layers.Dropout(0.1),  
    layers.Dense(64, activation='relu', 
                 kernel_regularizer=regularizers.l1_l2(l1=0.005, l2=0.005)),
    layers.Dropout(0.1),
    layers.Dense(64, activation='relu', 
                 kernel_regularizer=regularizers.l1_l2(l1=0.005, l2=0.005)),
    layers.Dense(1)  
])


In [ ]:
# Compilación del modelo
# Código aquí
model.compile(optimizer='adam', loss='mean_squared_error')

In [ ]:
## definir el early stopping callback
# Código aquí
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(monitor='val_loss',  
                               patience=10,        
                               restore_best_weights=True) 

model.fit(x_train,
          y_train,
          epochs=200,
          batch_size=32,
          validation_split=0.2,
          verbose=1,
          callbacks=early_stopping) # Código aquí

In [ ]:
# No modifique el código
results = model.evaluate(x_test, y_test, verbose=1)
print('Test Loss: {}'.format(results))

<a name='1.4'></a>
## Cuestión 4: ¿Podría haberse usado otra función de activación de la neurona de salida? En caso afirmativo especifíquela.

**Si**, *se podrian haber usado otras funciones de activación para la capa de salida, pero en este caso al ser un problema de regresión lineal, lo mas adecuado es utilizar la función lineal para la salida.*

**Pero las otras opciones son:**

***ReLU:*** Ideal para capas ocultas debido a su capacidad para introducir no linealidad en el modelo sin saturarse, pero no es adecuada para la salida ya que limita las predicciones a valores positivos.

***Tanh:*** Puede ser útil en capas ocultas si necesitas valores en el rango [-1, 1], pero no es adecuada para la salida de un modelo de regresión.

***Sigmoid:*** Útil para clasificación binaria o cuando las salidas están restringidas entre [0, 1], pero no es adecuada para una salida de regresión general.

<a name='1.5'></a>
## Cuestión 5:  ¿Qué es lo que una neurona calcula?

**a)** Una función de activación seguida de una suma ponderada  de las entradas.

**b)** Una suma ponderada  de las entradas seguida de una función de activación.

**c)** Una función de pérdida, definida sobre el target.

**d)** Ninguna  de las anteriores es correcta


La respuesta correcta es la **opción b**

<a name='1.6'></a>
## Cuestión 6:  ¿Cuál de estas funciones de activación no debería usarse en una capa oculta (hidden layer)?

**a)** `sigmoid`

**b)** `tanh`

**c)** `relu`

**d)** `linear`


La respuesta correcta es **d linear** 

<a name='1.7'></a>
## Cuestión 7:  ¿Cuál de estas técnicas es efectiva para combatir el overfitting en una red con varias capas ocultas? Ponga todas las que lo sean.

**a)** Dropout

**b)** Regularización L2.

**c)** Aumentar el tamaño del test set.

**d)** Aumentar el tamaño del validation set.

**e)** Reducir el número de capas de la red.

**f)** Data augmentation.


Respuestas correctas: 
**a) Dropout**,
**b) Regularización L2**,
**e) Reducir el número de capas de la red**,
**f) Data augmentation**

<a name='1.8'></a>
## Cuestión 8:  Supongamos que queremos entrenar una red para un problema de clasificación de imágenes con las siguientes clases: {'perro','gato','persona'}. ¿Cuántas neuronas y que función de activación debería tener la capa de salida? ¿Qué función de pérdida (loss function) debería usarse?


- Como estamos trabajando con un problema de clasificación multiclase (más de dos clases), la capa de salida debe tener una neurona por cada clase. En este caso, tenemos tres clases ('perro', 'gato', 'persona') por lo que la capa de salida debería tener **3 neuronas**.

- En un problema de clasificación multiclase, la función de activación más adecuada para la capa de salida es **softmax**.
  
    - *Softmax convierte las salidas en probabilidades, asegurándose de que la suma de las salidas de la capa de salida sea 1. Esto hace que cada neurona en la capa de salida represente la probabilidad de pertenecer a una de las clases.*
      

- Para un problema de clasificación multiclase con salida softmax, la función de pérdida más adecuada es **'categorical_crossentropy'**.

    - *La función 'categorical crossentropy' calcula la diferencia entre las probabilidades predichas y las clases reales (codificadas como vectores one-hot).*
    - 'perro' → [1, 0, 0] || 'gato' → [0, 1, 0] || 'persona' → [0, 0, 1]

<a name='actividad_2'></a>
# Actividad 2: Redes Convolucionales

Vamos a usar el dataset [cifar-10](https://www.cs.toronto.edu/~kriz/cifar.html), que son 60000 imágenes de 32x32 a color  con 10 clases diferentes. Para realizar mejor la práctica puede consultar [Introduction_to_CNN.ipynb](https://github.com/ezponda/intro_deep_learning/blob/main/class/CNN/Introduction_to_CNN.ipynb).



**Puntuación**: 

- [Cuestión 1](#2.1): 2.5 pt
- [Cuestión 2](#2.2): 1 pt
- [Cuestión 3](#2.3): 0.5 pts
- [Cuestión 4](#2.4): 0.5 pts
- [Cuestión 5](#2.5): 0.5 pts

Puede normalizar las imágenes al principio o usar la capa [Rescaling](https://www.tensorflow.org/api_docs/python/tf/keras/layers/experimental/preprocessing/Rescaling):

```python
tf.keras.layers.experimental.preprocessing.Rescaling(
    scale, offset=0.0, name=None, **kwargs
)
```

In [ ]:

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
y_train = y_train.flatten()
y_test = y_test.flatten()

In [ ]:

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

plt.figure(figsize=(10,10))
for i in range(25):
    plt.subplot(5,5,i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(x_train[i])
    plt.xlabel(class_names[y_train[i]])
plt.show()

In [ ]:

print('x_train, y_train shapes:', x_train.shape, y_train.shape)
print('x_test, y_test shapes:', x_test.shape, y_test.shape)

<a name='2.1'></a>
## Cuestión 1: Cree una red convolucional con la API funcional con al menos dos capas convolucionales y al menos dos capas de pooling. Debe obtener un `Test accuracy > 0.68`

In [ ]:
# Definición del modelo usando la API funcional de Keras
inputs = tf.keras.Input(shape=(32, 32, 3), name='input')

# Capa de rescaling nombrada (sin 'experimental')
reescaling = layers.Rescaling(1./255)(inputs)

# Convolution + pooling layers
x = layers.Conv2D(64, (3, 3), activation='relu')(reescaling)
x = layers.MaxPooling2D(pool_size=(2, 2))(x)

# Otra capa convolucional + MaxPooling
x = layers.Conv2D(64, (3, 3), activation='relu')(x)
x = layers.MaxPooling2D(pool_size=(2, 2))(x)

# Dropout para reducir el sobreajuste
x = layers.Dropout(0.3)(x)

# Flattening
x = layers.Flatten()(x)

# Fully-connected
x = layers.Dense(128, activation='relu')(x)

outputs = layers.Dense(10, activation='softmax')(x)

model = models.Model(inputs=inputs, outputs=outputs)


In [ ]:
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])

In [ ]:
history = model.fit(x_train, y_train, epochs=25, batch_size=64,
                    validation_split=0.15)

In [ ]:
results = model.evaluate(x_test, y_test, verbose=0, batch_size=1000)
print('Test Loss: {}'.format(results[0]))
print('Test Accuracy: {}'.format(results[1]))

<a name='2.2'></a>
## Cuestión 2: Cree el mismo  modelo de manera secuencial. No es necesario compilar ni entrenar el modelo

In [ ]:

model_seq = tf.keras.models.Sequential([
    layers.Rescaling(1./255, input_shape=(32, 32, 3)),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),

    layers.Dropout(0.3),
    layers.Flatten(),

    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

# Resumen del modelo
model_seq.summary()

<a name='2.3'></a>
## Cuestión 3: Si tenenemos una  una imagen de entrada de 300 x 300 a color (RGB) y queremos usar una red densa. Si la primera capa oculta tiene 100 neuronas, ¿Cuántos parámetros tendrá esa capa (sin incluir el bias) ?


Para calcular el número de parámetros en una capa densa (fully connected layer) de una red neuronal, se usa la siguiente fórmula:

Numero de parametros = (Numero de entradas) × (Numero de neuronas en la capa oculta) 

En este caso:
Imagen de entrada de 300x300 a color (RGB): La imagen tiene dimensiones de 300x300 píxeles y 3 canales RGB, por lo que el número total de entradas a la capa densa es:

        300 × 300 × 3 = 270.000

Primera capa oculta con 100 neuronas: El número de neuronas en la capa oculta es 100.

Cálculo numero de parametros: 

        270.000 × 100 = 27.000.000

***Por lo tanto, el número total de parámetros en la primera capa es 27,000,000.***


<a name='2.4'></a>
## Cuestión 4   Ponga  las verdaderas ventajas de las redes convolucionales respecto a las densas

**a)** Reducen el número total de parámetros, reduciendo así el overfitting.

**b)** Permiten utilizar una misma 'función'  en varias localizaciones de la imagen de entrada, en lugar de aprender una función diferente para cada pixel.

**c)** Permiten el uso del transfer learning.

**d)** Generalmente son menos profundas, lo que facilita su entrenamiento.



**La respuestas correctas son a), b) y c)**

<a name='2.5'></a>
## Cuestión 5: Para el procesamiento de series temporales las redes convolucionales no son efectivas, habrá que usar redes recurrentes.

- **Verdadero** 
- **Falso** 

**Esta afrimación es <ins>*Falsa*<ins>.** 

*Aunque es cierto que las arquitecturas mas populares para series temporales son las RNN o LSTM, pero las CNN tambien pueden ser efectivas, especialmente cuando se combinan con otros modelos.*